# Level 8: Quantum Phase Estimation & Shor's Algorithm
## Qiskit Edition

In this notebook, you will:

1. Implement **Quantum Phase Estimation (QPE)** — the heart of many quantum algorithms
2. Understand how QPE leads to **Shor's algorithm** for factoring
3. Build QPE for the T gate and find its eigenphase
4. See the implications for RSA cryptography

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt
import numpy as np
from fractions import Fraction

simulator = AerSimulator()
print("Ready!")

---
## 1. Quantum Phase Estimation (QPE)

**Problem**: Given a unitary $U$ and its eigenvector $|\psi\rangle$ where $U|\psi\rangle = e^{2\pi i \theta}|\psi\rangle$, find $\theta$.

### Why is this useful?
- Finding eigenvalues of quantum operators
- Period finding (Shor's algorithm)
- Quantum chemistry (energy estimation)

### The Algorithm
1. Use $n$ **counting qubits** in superposition
2. Apply controlled-$U^{2^k}$ for $k = 0, 1, \ldots, n-1$
3. Apply **inverse QFT** to the counting qubits
4. Measure → get an $n$-bit approximation of $\theta$

In [ ]:
def inverse_qft(qc, n):
    """Apply inverse QFT to the first n qubits of circuit qc."""
    # Swap qubits
    for i in range(n // 2):
        qc.swap(i, n - 1 - i)
    
    # Reverse QFT
    for i in range(n - 1, -1, -1):
        for j in range(n - 1, i, -1):
            k = j - i + 1
            angle = -2 * np.pi / (2**k)
            qc.cp(angle, j, i)
        qc.h(i)

print("Inverse QFT defined!")

---
## 2. QPE for the T Gate

The T gate has: $T|1\rangle = e^{i\pi/4}|1\rangle = e^{2\pi i \cdot 1/8}|1\rangle$

So $\theta = 1/8 = 0.125$. With 3 counting qubits we should measure $|001\rangle$ (binary 001 = 1, so $\theta = 1/8$).

In [ ]:
def qpe_for_t_gate(n_counting):
    """QPE to estimate the phase of the T gate."""
    # n counting qubits + 1 eigenstate qubit
    counting = QuantumRegister(n_counting, 'counting')
    eigenstate = QuantumRegister(1, 'eigenstate')
    classical = ClassicalRegister(n_counting, 'c')
    
    qc = QuantumCircuit(counting, eigenstate, classical)
    
    # Prepare eigenstate |1>
    qc.x(eigenstate[0])
    
    # Step 1: Put counting qubits in superposition
    qc.h(counting)
    qc.barrier()
    
    # Step 2: Controlled-U^(2^k) operations
    for k in range(n_counting):
        # T^(2^k) is a phase rotation of pi/4 * 2^k
        angle = np.pi / 4 * (2**k)
        qc.cp(angle, counting[k], eigenstate[0])
    qc.barrier()
    
    # Step 3: Inverse QFT
    inverse_qft(qc, n_counting)
    qc.barrier()
    
    # Step 4: Measure counting qubits
    qc.measure(counting, classical)
    
    return qc

# 3-bit QPE for T gate
qpe3 = qpe_for_t_gate(3)
print("QPE Circuit for T gate (3 counting qubits):")
qpe3.draw('mpl')

In [ ]:
result = simulator.run(qpe3, shots=1000).result().get_counts()
print(f"QPE Results: {result}")

# Convert the measured bits to the estimated phase
for bitstring, count in result.items():
    value = int(bitstring, 2)
    theta = value / (2**3)
    print(f"  |{bitstring}> = {value}/8 → θ = {theta:.4f} (count: {count})")

print(f"\nExpected: θ = 1/8 = 0.125")
plot_histogram(result, title="QPE for T gate")

---
## 3. QPE with More Precision

Let's try with a gate where the phase is NOT an exact fraction of $2^n$.

In [ ]:
def qpe_for_phase(theta_real, n_counting):
    """QPE for a gate with eigenphase theta_real (as fraction of 2*pi)."""
    counting = QuantumRegister(n_counting, 'counting')
    eigenstate = QuantumRegister(1, 'eigenstate')
    classical = ClassicalRegister(n_counting, 'c')
    
    qc = QuantumCircuit(counting, eigenstate, classical)
    qc.x(eigenstate[0])
    qc.h(counting)
    qc.barrier()
    
    for k in range(n_counting):
        angle = 2 * np.pi * theta_real * (2**k)
        qc.cp(angle, counting[k], eigenstate[0])
    qc.barrier()
    
    inverse_qft(qc, n_counting)
    qc.measure(counting, classical)
    
    return qc

# QPE for theta = 1/3 (not an exact binary fraction)
qc = qpe_for_phase(1/3, 5)
result = simulator.run(qc, shots=4000).result().get_counts()

print("QPE for θ = 1/3 (5 counting qubits):")
# Show top results
sorted_results = sorted(result.items(), key=lambda x: x[1], reverse=True)
for bits, count in sorted_results[:5]:
    value = int(bits, 2)
    theta_est = value / 32
    frac = Fraction(value, 32).limit_denominator(10)
    print(f"  |{bits}> = {value}/32 → θ ≈ {theta_est:.4f} ≈ {frac} (count: {count})")

print(f"\nExpected: θ = 1/3 ≈ 0.3333")
plot_histogram(result, title="QPE for θ = 1/3")

---
## 4. From QPE to Shor's Algorithm

### The Connection

Shor's algorithm factors $N$ by:
1. Choose random $a < N$ with $\gcd(a, N) = 1$
2. Use QPE to find the **period** $r$ of $f(x) = a^x \mod N$
3. If $r$ is even, then $\gcd(a^{r/2} \pm 1, N)$ gives factors of $N$

### Why This Breaks RSA
- RSA security relies on factoring being hard: $O(e^{n^{1/3}})$ classically
- Shor's runs in $O(n^3)$ — polynomial time!
- For RSA-2048: classical would take billions of years; Shor's on a quantum computer could do it in hours

### Simplified Demo: Factoring 15

Let's factor $N = 15$ using $a = 7$. We need to find the period of $7^x \mod 15$.

In [ ]:
# Classical verification: find the period of 7^x mod 15
a, N = 7, 15
print(f"Finding period of {a}^x mod {N}:")
for x in range(10):
    val = pow(a, x, N)
    print(f"  {a}^{x} mod {N} = {val}")

print(f"\nPeriod r = 4 (pattern repeats: 1, 7, 4, 13, 1, 7, ...)")
print(f"\nFactors: gcd({a}^({4}//2) - 1, {N}) = gcd({a**2 - 1}, {N}) = {np.gcd(a**2 - 1, N)}")
print(f"         gcd({a}^({4}//2) + 1, {N}) = gcd({a**2 + 1}, {N}) = {np.gcd(a**2 + 1, N)}")
print(f"\n15 = 3 × 5  ✓")

In [ ]:
# Simplified Shor's: Period-finding for 7^x mod 15 using QPE
# We use 3 counting qubits + 4 work qubits

def controlled_mod_multiply(qc, control, target_qubits, a, N):
    """Simplified controlled modular multiplication for small cases."""
    # For a=7, N=15: specific implementation
    if a == 7 and N == 15:
        qc.cswap(control, target_qubits[0], target_qubits[1])
        qc.cswap(control, target_qubits[1], target_qubits[2])
        qc.cswap(control, target_qubits[2], target_qubits[3])
        # Flip bits for mod operation
        for q in target_qubits:
            qc.cx(control, q)

def shors_period_finding(a, N, n_counting=3):
    """Simplified Shor's period-finding circuit."""
    n_work = 4  # Enough for N=15
    counting = QuantumRegister(n_counting, 'counting')
    work = QuantumRegister(n_work, 'work')
    classical = ClassicalRegister(n_counting, 'c')
    
    qc = QuantumCircuit(counting, work, classical)
    
    # Initialize work register to |1>
    qc.x(work[0])
    
    # Hadamard on counting qubits
    qc.h(counting)
    qc.barrier()
    
    # Controlled modular exponentiation
    for k in range(n_counting):
        power = 2**k
        a_power = pow(a, power, N)
        # Apply controlled-U^(2^k)
        for _ in range(power):
            controlled_mod_multiply(qc, counting[k], list(range(n_counting, n_counting + n_work)), a, N)
    qc.barrier()
    
    # Inverse QFT
    inverse_qft(qc, n_counting)
    
    # Measure
    qc.measure(counting, classical)
    
    return qc

qc_shor = shors_period_finding(7, 15, 3)
result = simulator.run(qc_shor, shots=4000).result().get_counts()

print("Shor's period finding for 7^x mod 15:")
for bits, count in sorted(result.items(), key=lambda x: -x[1]):
    value = int(bits, 2)
    phase = value / 8
    frac = Fraction(phase).limit_denominator(15)
    print(f"  |{bits}> = {value}/8 → phase = {phase:.3f} → {frac} (count: {count})")

plot_histogram(result, title="Shor's: Period finding for 7^x mod 15")

---
## 5. Key Takeaways

| Concept | Description |
|---------|-------------|
| **QPE** | Finds eigenphase θ using controlled-U and inverse QFT |
| **Precision** | n counting qubits → n-bit precision |
| **Shor's** | Uses QPE for period finding → factoring in O(n³) |
| **RSA impact** | Shor's would break RSA if large quantum computers existed |

---
## 6. Challenges

1. **QPE for S gate**: The S gate has phase $1/4$. Run QPE with 3 bits to verify.
2. **More precision**: Run QPE for θ=1/3 with 8 counting qubits. How close is the answer?
3. **Factor 21**: Adapt the Shor's circuit to factor 21 = 3 × 7. Pick a=2 and find the period.

In [ ]:
# Your challenge code here!

---
**Next up: [Level 9 — NISQ and Hybrid Algorithms](../Level_09_NISQ_Hybrid/Level_09_Qiskit.ipynb)**